# CSV to Postgres ETL

This notebook is the actual data engineering workflow for our project:

1. **Extract** — read the business's raw, messy CSV.
2. **Explore** — look at what's actually wrong with it.
3. **Clean** — fix the issues we found.
4. **Split** — break the one wide CSV into the 7 normalized tables.
5. **Load** — push each table into Postgres, in the right order.
6. **Verify** — query Postgres to confirm everything landed correctly.

## Imports and Setup

In [2]:
# !pip install PyYAML

In [3]:
import pandas as pd # for data ingestion, transformation and exploration
import yaml # for reading the configuration file with our database credentials
import sqlalchemy # for connecting to the database and executing SQL queries

# For Visibiltiy
pd.set_option("display.max_columns", 30) # for better visibility of all columns
pd.set_option("display.width", 150) # for better visibility of all columns


### 1. **Extract** — read the business's raw, messy CSV.

In [4]:
raw_data_path = "../data/raw_logistics_export.csv"

date_columns = ["order_date", "ship_date", "expected_delivery_date", "actual_delivery_date"]

df = pd.read_csv(raw_data_path, parse_dates=date_columns)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Rows: 1279292
Columns: 29


,order_id,order_date,order_status,customer_id,customer_name,email,customer_city,customer_region,customer_country,product_id,product_name,category,unit_price,order_item_id,quantity,line_total,warehouse_id,warehouse_name,warehouse_city,warehouse_region,shipment_id,carrier_id,carrier_name,service_level,ship_date,expected_delivery_date,actual_delivery_date,delivery_status,shipping_cost
0,123126,2024-07-07,Completed,2233,Robert Jones,coxsamantha@example.net,Nalerigu,North East,Ghana,233,User-friendly zero tolerance project,Electronics,680.13,313370,1,680.13,13,Kasoa Distribution Center,Kasoa,Central,123126,4,Meridian Shipping,Freight,2024-07-08,2024-07-13,2024-07-16,Delayed,183.03
1,494084,2025-08-25,Completed,4560,Craig Roach,mckinneyjennifer@example.net,Tarkwa,Western,Ghana,221,Cross-platform secondary attitude,Toys & Games,696.24,1257774,3,2088.72,10,Tarkwa Distribution Center,Tarkwa,Western,494084,6,Coastal Express,Overnight,2025-08-27,2025-09-01,2025-09-03,Delayed,82.30
2,211063,2025-06-13,Completed,946,Joshua Gonzales,jacob61@example.net,CAPE COAST,Central,Ghana,362,Focused bi-directional groupware,Home & Garden,821.12,537466,7,5747.84,1,Sunyani Distribution Center,Sunyani,Bono,211063,5,Pioneer Transport,Standard,2025-06-14,2025-06-15,2025-06-17,Delayed,194.75
3,192331,2024-11-29,Processing,785,Kevin Jenkins,kingdean@example.org,Akosombo,Eastern,Ghana,499,Extended client-driven framework,Furniture,44.29,489770,7,310.03,1,Sunyani Distribution Center,Sunyani,Bono,192331,4,Meridian Shipping,Standard,2024-11-30,2024-12-01,NaT,In Transit,202.53
4,447985,2026-02-24,Completed,2644,Brooke Pierce,audreyrogers@example.org,Sefwi Wiawso,Western North,Ghana,298,Profit-focused real-time solution,Apparel,287.52,1140622,9,2587.68,8,Sekondi-Takoradi Distribution Center,Sekondi-Takoradi,Western,447985,6,Coastal Express,Overnight,2026-02-24,2026-02-28,2026-03-03,Delayed,216.70


### 2. **Explore** — Find missing values, duplicates, and other data quality issues.

In [5]:
# How many duplicate rows do we have?
num_dup_rows = df.duplicated().sum()
print(f"Duplicate Rows: {num_dup_rows}")

Duplicate Rows: 5694


In [6]:
missing_data_perc = (df.isna().sum().sum()/len(df)) * 100
print(f"Percenatage of missing data: {missing_data_perc:.2f}%")

Percenatage of missing data: 17.32%


In [7]:
# How many missing values are in each column
df.isna().sum()

order_id                       0
order_date                     0
order_status                   0
customer_id                    0
customer_name                  0
email                      12793
customer_city                  0
customer_region                0
customer_country               0
product_id                     0
product_name                   0
category                       0
unit_price                     0
order_item_id                  0
quantity                       0
line_total                     0
warehouse_id                   0
warehouse_name                 0
warehouse_city                 0
warehouse_region               0
shipment_id                    0
carrier_id                     0
carrier_name                   0
service_level                  0
ship_date                      0
expected_delivery_date         0
actual_delivery_date      208785
delivery_status                0
shipping_cost                  0
dtype: int64

In [8]:
# Inconsistent text casing in customer city
df["customer_city"].drop_duplicates().head(20)

0             Nalerigu
1               Tarkwa
2           CAPE COAST
3             Akosombo
4         Sefwi Wiawso
5                Bawku
6         SEFWI WIAWSO
7                Lawra
8             AKOSOMBO
10             Damongo
11          Cape Coast
12           Koforidua
14              OBUASI
17                KETA
19               Goaso
24             Berekum
25             Sunyani
26    SEKONDI-TAKORADI
27               LAWRA
28               ACCRA
Name: customer_city, dtype: object

### 3. **Clean** — fix the issues we found.

In [9]:
# Drop Duplicate rows
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)

print(f"Removed {rows_before - rows_after} duplicate rows")

Removed 5694 duplicate rows


In [10]:
# Strip extra whitespaces
text_columns = df.select_dtypes(include=['object', 'string']).columns.tolist()

for column in text_columns:
    df[column] = df[column].astype(str).str.strip()
    print(f"Stripped whitespaces from {column}")

print("Done!")

Stripped whitespaces from order_status
Stripped whitespaces from customer_name
Stripped whitespaces from email
Stripped whitespaces from customer_city
Stripped whitespaces from customer_region
Stripped whitespaces from customer_country
Stripped whitespaces from product_name
Stripped whitespaces from category
Stripped whitespaces from warehouse_name
Stripped whitespaces from warehouse_city
Stripped whitespaces from warehouse_region
Stripped whitespaces from carrier_name
Stripped whitespaces from service_level
Stripped whitespaces from delivery_status
Done!


In [11]:
# Standardize city name casing so "ACCRA" will look like "Accra" 
df["customer_city"] = df["customer_city"].str.title()

# to check
df["customer_city"].drop_duplicates().head(20)

0             Nalerigu
1               Tarkwa
2           Cape Coast
3             Akosombo
4         Sefwi Wiawso
5                Bawku
7                Lawra
10             Damongo
12           Koforidua
14              Obuasi
17                Keta
19               Goaso
24             Berekum
25             Sunyani
26    Sekondi-Takoradi
28               Accra
29            Ashaiman
33              Tamale
35               Ejisu
38            Kintampo
Name: customer_city, dtype: object

In [12]:
# Fill up missing values in the email column
df['email'] = df['email'].fillna("Not Given")

df["email"].isna().sum()

np.int64(0)

### 4. **Split** — break the one wide CSV into the 7 normalized tables.

This part builds each table by picking the relevant columns out of the `df`, then using `drop_duplicates()` so each table only keeps **one row per entity**
(e.g. one row per warehouse, not one row per warehouse per order anymore).

In [13]:
# warehouses table
warehouse_columns = ["warehouse_id", "warehouse_name", "warehouse_city", "warehouse_region"]

# drop duplicates
warehouse_table = df[warehouse_columns].drop_duplicates(subset="warehouse_id")
# Sort by warehouse id
warehouse_table = warehouse_table.sort_values("warehouse_id")

print(f"Warehouses {len(warehouse_table)} rows")
warehouse_table.head(15)

Warehouses 15 rows


,warehouse_id,warehouse_name,warehouse_city,warehouse_region
2,1,Sunyani Distribution Center,Sunyani,Bono
9,2,Yendi Distribution Center,Yendi,Northern
26,3,Konongo Distribution Center,Konongo,Ashanti
11,4,Sefwi Wiawso Distribution Center,Sefwi Wiawso,Western North
17,5,Tarkwa Distribution Center,Tarkwa,Western
8,6,Teshie Distribution Center,Teshie,Greater Accra
13,7,Bolgatanga Distribution Center,Bolgatanga,Upper East
4,8,Sekondi-Takoradi Distribution Center,Sekondi-Takoradi,Western
7,9,Sefwi Wiawso Distribution Center,Sefwi Wiawso,Western North
1,10,Tarkwa Distribution Center,Tarkwa,Western


In [ ]:
# carriers table
carrier_columns = ["carrier_id", "carrier_name"]
carriers_table = df[carrier_columns].drop_duplicates(subset="carrier_id")
carriers_table = carriers_table.sort_values("carrier_id")

print("carriers:", len(carriers_table), "rows")
carriers_table.head()


carriers: 8 rows


,carrier_id,carrier_name
5,1,SwiftHaul Logistics
18,2,BlueArrow Freight
6,3,Nexus Cargo
0,4,Meridian Shipping
2,5,Pioneer Transport


In [ ]:
# products table
product_columns = ["product_id", "product_name", "category", "unit_price"]
products_table = df[product_columns].drop_duplicates(subset="product_id")
products_table = products_table.sort_values("product_id")

print(f"products: {len(products_table)} rows")
products_table.head()

products: 500 rows


,product_id,product_name,category,unit_price
555,1,Quality-focused explicit pricing structure,Food & Beverage,430.54
382,2,Assimilated regional projection,Food & Beverage,259.35
313,3,Expanded eco-centric application,Industrial Equipment,293.38
460,4,Object-based homogeneous project,Health & Beauty,106.91
61,5,Devolved attitude-oriented Local Area Network,Home & Garden,621.03


In [ ]:
# customers table
customer_columns = ["customer_id", "customer_name", "email", "customer_city", "customer_region", "customer_country"]
customers_table = df[customer_columns].drop_duplicates(subset="customer_id")
customers_table = customers_table.sort_values("customer_id")

print(f"customers: {len(customers_table)} rows")
customers_table.head()

customers: 5000 rows


,customer_id,customer_name,email,customer_city,customer_region,customer_country
4749,1,Diana Martin,ycarr@example.net,Nkawkaw,Eastern,Ghana
3250,2,Keith Rodriguez,rowebrenda@example.net,Berekum,Bono,Ghana
7801,3,Cynthia Roberts,melissagraham@example.org,Sekondi-Takoradi,Western,Ghana
8993,4,Thomas Garza,cwood@example.net,Bawku,Upper East,Ghana
1738,5,Susan Green,stephanie22@example.org,Techiman,Bono East,Ghana


In [ ]:
# orders table
order_columns = ["order_id", "customer_id", "warehouse_id", "order_date", "order_status"]
orders_table = df[order_columns].drop_duplicates(subset="order_id")
orders_table = orders_table.sort_values("order_id")

print(f"orders: {len(orders_table)} rows")
orders_table.head()

orders: 500000 rows


,order_id,customer_id,warehouse_id,order_date,order_status
887783,1,3038,5,2024-02-10,Completed
720622,2,4840,8,2025-09-08,Completed
458165,3,333,4,2026-05-27,Completed
589602,4,459,11,2024-09-26,Completed
441910,5,3092,10,2024-03-20,Completed


In [ ]:
# order_items table
order_item_columns = ["order_item_id", "order_id", "order_date", "product_id", "quantity", "unit_price", "line_total"]
order_items_table = df[order_item_columns].drop_duplicates(subset="order_item_id")
order_items_table = order_items_table.sort_values("order_item_id")

print(f"order_items: {len(order_items_table)} rows")
order_items_table.head()


order_items: 1272927 rows


,order_item_id,order_id,order_date,product_id,quantity,unit_price,line_total
887783,1,1,2024-02-10,102,4,157.22,628.88
982834,2,2,2025-09-08,387,9,276.13,2485.17
720622,3,2,2025-09-08,316,6,553.23,3319.38
458165,4,3,2026-05-27,9,3,814.43,2443.29
589602,5,4,2024-09-26,107,4,847.76,3391.04


In [ ]:
# shipments table
shipment_columns = ["shipment_id", "order_id", "carrier_id", "service_level", "ship_date","expected_delivery_date", "actual_delivery_date", "delivery_status", "shipping_cost"]
shipments_table = df[shipment_columns].drop_duplicates(subset="shipment_id")
shipments_table = shipments_table.sort_values("shipment_id")

print(f"shipments: {len(shipments_table)} rows")
shipments_table.head()

shipments: 500000 rows


,shipment_id,order_id,carrier_id,service_level,ship_date,expected_delivery_date,actual_delivery_date,delivery_status,shipping_cost
887783,1,1,7,Standard,2024-02-12,2024-02-14,NaT,In Transit,209.47
720622,2,2,6,Standard,2025-09-10,2025-09-16,2025-09-16,Delivered,62.47
458165,3,3,8,Express,2026-05-28,2026-06-06,2026-06-07,Delayed,241.94
589602,4,4,6,Standard,2024-09-28,2024-10-01,2024-10-03,Delayed,237.65
441910,5,5,2,Standard,2024-03-21,2024-03-29,2024-03-29,Delivered,57.53


### 5. **Load** — push each table into Postgres, in the right order.

**Order matters here** because of foreign key constraints: a table can
only be loaded once the tables it references already have their rows in
place.

Load order: `warehouses` → `carriers` → `products` → `customers`
(no dependencies) → `orders` (needs customers + warehouses) →
`order_items` and `shipments` (both need orders).

We use `to_sql(..., if_exists="append")`because we have already
created the empty tables (and partitions) for us; we're only inserting
rows, not creating tables from pandas.

### Database Connection Setting

In [ ]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

    host = config.get('host')
    user = config.get('user')
    database_name = config.get('database')
    password = config.get('password')
    port = config.get('port')

# build the URl 
db_url = sqlalchemy.URL.create(
    drivername='postgresql+psycopg2',
    username=user,
    host=host,
    database=database_name,
    password=password,
    port=port
)

# build the engine
engine = sqlalchemy.create_engine(db_url)
print(f"Connection Ready! {db_url}")

In [ ]:
# Load the warehouse table - to be executed once
warehouse_table.to_sql(name="warehouses", con=engine, if_exists="append",schema="logistics", index=False)

print(f"Loaded {len(warehouse_table)} to the database {database_name}")

Loaded 15 to the database sankofa_db


In [29]:
# Load the carriers table - to be executed once
carriers_table.to_sql(name="carriers", con=engine, if_exists="append",schema="logistics", index=False)

print(f"Loaded {len(carriers_table)} carries to the database {database_name}")

Loaded 8 carries to the database sankofa_db


In [30]:
# Load the products table - to be executed once
products_table.to_sql(name="products", con=engine, if_exists="append",schema="logistics", index=False)

print(f"Loaded {len(products_table)} products to the database {database_name}")

Loaded 500 products to the database sankofa_db


In [ ]:
# Load the customers table - to be executed once
customers_table.to_sql(name="customers", con=engine, if_exists="append",schema="logistics", index=False)

print(f"Loaded {len(customers_table)} customers to the database {database_name}")

Loaded 5000 products to the database sankofa_db


In [32]:
# Load the orders table - to be executed once
orders_table.to_sql(name="orders", con=engine, if_exists="append",schema="logistics", index=False, method='multi', chunksize=5000)

print(f"Loaded {len(orders_table)} orders to the database {database_name}")

Loaded 500000 orders to the database sankofa_db


In [33]:
# Load the order_items table - to be executed once
order_items_table.to_sql(name="order_items", con=engine, if_exists="append",schema="logistics", index=False, method='multi', chunksize=10000)

print(f"Loaded {len(order_items_table)} order_items to the database {database_name}")

Loaded 1272927 order_items to the database sankofa_db


In [34]:
# Load the shipments table - to be executed once
shipments_table.to_sql(name="shipments", con=engine, if_exists="append",schema="logistics", index=False, method='multi', chunksize=5000)

print(f"Loaded {len(shipments_table)} shipments to the database {database_name}")

Loaded 500000 shipments to the database sankofa_db


### **Verify** — query Postgres to confirm everything landed correctly.

In [ ]:
table_names = ["warehouses", "carriers", "products", "customers", "orders", "order_items", "shipments"]

for table_name in table_names:
    result = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM logistics.{table_name}", con=engine)
    print(table_name, "->", result["row_count"].iloc[0], "rows")


warehouses -> 15 rows
carriers -> 8 rows
products -> 500 rows
customers -> 5000 rows
orders -> 500000 rows
order_items -> 1272927 rows
shipments -> 500000 rows
